<a href="https://colab.research.google.com/github/shrutimali25/CareFlow/blob/madhura/notebooks/CareFlow_Process_Mining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CareFlow — Clinical Pathway Process Mining

## Purpose
Analyze hospital event logs to understand patient journeys, discover process flows, and investigate elapsed times between activities.

## Analysis Plan
1. Inspect and validate the source data.
2. Understand the event-log structure.
3. Discover process maps using PM4Py.
4. Analyze common pathways and repeated activities.
5. Calculate elapsed times between activities.
6. Prepare validated outputs for Power BI.
7. Document findings and limitations.

## Data Integrity
All reported results must be calculated from the actual source data and validated before reporting.

In [2]:
import sys
from datetime import datetime

print("CareFlow environment initialized")
print("Python version:", sys.version.split()[0])
print("Notebook started:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

CareFlow environment initialized
Python version: 3.13.18
Notebook started: 2026-11-10 17:09:35


In [3]:
from google.colab import files

uploaded = files.upload()

print("\nFiles uploaded:")
for filename in uploaded:
    print("-", filename)

Saving activity_frequency.csv to activity_frequency.csv
Saving activity_time_differences.csv to activity_time_differences.csv
Saving patient_case_summary.csv to patient_case_summary.csv

Files uploaded:
- activity_frequency.csv
- activity_time_differences.csv
- patient_case_summary.csv


In [4]:
import pandas as pd
from pathlib import Path

# CareFlow: load the three analysis files
data_files = {
    "patient_cases": "patient_case_summary.csv",
    "activity_frequency": "activity_frequency.csv",
    "activity_times": "activity_time_differences.csv"
}

data = {}

for name, filename in data_files.items():
    path = Path(filename)

    if not path.exists():
        print(f"File not found: {filename}")
        continue

    data[name] = pd.read_csv(path)
    df = data[name]

    print(f"\n{'=' * 55}")
    print(f"FILE: {filename}")
    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")
    print("Column names:", df.columns.tolist())
    print("\nMissing values per column:")
    print(df.isna().sum().to_string())

print("\nCareFlow file inspection completed.")


FILE: patient_case_summary.csv
Rows: 1049
Columns: 5
Column names: ['case:concept:name', 'total_events', 'first_event_time', 'last_event_time', 'treatment_duration_hours']

Missing values per column:
case:concept:name           0
total_events                0
first_event_time            0
last_event_time             0
treatment_duration_hours    0

FILE: activity_frequency.csv
Rows: 16
Columns: 3
Column names: ['concept:name', 'total_occurrences', 'unique_cases']

Missing values per column:
concept:name         0
total_occurrences    0
unique_cases         0

FILE: activity_time_differences.csv
Rows: 115
Columns: 5
Column names: ['previous_activity', 'concept:name', 'total_transitions', 'average_minutes', 'median_minutes']

Missing values per column:
previous_activity    0
concept:name         0
total_transitions    0
average_minutes      0
median_minutes       0

CareFlow file inspection completed.


In [5]:

import pandas as pd
from pathlib import Path

data_files = {
    "patient_cases": "patient_case_summary.csv",
    "activity_frequency": "activity_frequency.csv",
    "activity_times": "activity_time_differences.csv"
}

data = {}

for name, filename in data_files.items():
    if not Path(filename).exists():
        raise FileNotFoundError(
            f"{filename} is missing. Please upload the CSV files again."
        )
    data[name] = pd.read_csv(filename)

print("All three CareFlow CSV files loaded successfully.")

for name, df in data.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")

All three CareFlow CSV files loaded successfully.
patient_cases: 1049 rows, 5 columns
activity_frequency: 16 rows, 3 columns
activity_times: 115 rows, 5 columns


In [6]:
# CareFlow: Data Quality and Consistency Validation

cases = data["patient_cases"].copy()
frequency = data["activity_frequency"].copy()
transitions = data["activity_times"].copy()

# 1. Validate patient case IDs
print("1. CASE ID VALIDATION")
print("Duplicate case IDs:",
      cases["case:concept:name"].duplicated().sum())
print("Unique case IDs:",
      cases["case:concept:name"].nunique())

# 2. Compare event totals
case_event_total = pd.to_numeric(
    cases["total_events"], errors="coerce"
).sum()

activity_event_total = pd.to_numeric(
    frequency["total_occurrences"], errors="coerce"
).sum()

print("\n2. EVENT COUNT VALIDATION")
print("Events in case summary:", int(case_event_total))
print("Events in activity frequency:", int(activity_event_total))
print("Event totals match:", case_event_total == activity_event_total)

# 3. Validate activity counts
print("\n3. ACTIVITY FREQUENCY VALIDATION")
print("Distinct activities:", frequency["concept:name"].nunique())

print("Negative activity occurrence counts:",
      (pd.to_numeric(frequency["total_occurrences"],
                     errors="coerce") < 0).sum())

print("Activities with unique-case counts exceeding total cases:",
      (pd.to_numeric(frequency["unique_cases"],
                     errors="coerce")
       > cases["case:concept:name"].nunique()).sum())

# 4. Validate timestamps and visit durations
print("\n4. TIMESTAMP AND DURATION VALIDATION")

cases["first_event_time"] = pd.to_datetime(
    cases["first_event_time"], errors="coerce", utc=True
)
cases["last_event_time"] = pd.to_datetime(
    cases["last_event_time"], errors="coerce", utc=True
)
cases["treatment_duration_hours"] = pd.to_numeric(
    cases["treatment_duration_hours"], errors="coerce"
)

invalid_timestamps = (
    cases["first_event_time"].isna()
    | cases["last_event_time"].isna()
)

negative_elapsed = (
    cases["last_event_time"] < cases["first_event_time"]
).fillna(False)

calculated_hours = (
    cases["last_event_time"] - cases["first_event_time"]
).dt.total_seconds() / 3600

duration_difference = (
    calculated_hours - cases["treatment_duration_hours"]
).abs()

print("Invalid timestamps:", int(invalid_timestamps.sum()))
print("Last event before first event:", int(negative_elapsed.sum()))
print("Missing or invalid durations:",
      int(cases["treatment_duration_hours"].isna().sum()))
print("Negative durations:",
      int((cases["treatment_duration_hours"] < 0).sum()))
print("Durations differing from timestamps by over 1 minute:",
      int((duration_difference > 1/60).sum()))

# 5. Validate transition data
print("\n5. TRANSITION VALIDATION")

transition_counts = pd.to_numeric(
    transitions["total_transitions"], errors="coerce"
)
average_minutes = pd.to_numeric(
    transitions["average_minutes"], errors="coerce"
)
median_minutes = pd.to_numeric(
    transitions["median_minutes"], errors="coerce"
)

print("Transition pairs:", len(transitions))
print("Invalid or non-positive transition counts:",
      int(((transition_counts <= 0) |
           transition_counts.isna()).sum()))
print("Negative average times:",
      int((average_minutes < 0).sum()))
print("Negative median times:",
      int((median_minutes < 0).sum()))
print("Rows with missing transition metrics:",
      int(transitions[
          ["total_transitions", "average_minutes", "median_minutes"]
      ].isna().any(axis=1).sum()))

print("\nCareFlow validation checks completed.")

1. CASE ID VALIDATION
Duplicate case IDs: 0
Unique case IDs: 1049

2. EVENT COUNT VALIDATION
Events in case summary: 15189
Events in activity frequency: 15189
Event totals match: True

3. ACTIVITY FREQUENCY VALIDATION
Distinct activities: 16
Negative activity occurrence counts: 0
Activities with unique-case counts exceeding total cases: 0

4. TIMESTAMP AND DURATION VALIDATION
Invalid timestamps: 0
Last event before first event: 0
Missing or invalid durations: 0
Negative durations: 0
Durations differing from timestamps by over 1 minute: 0

5. TRANSITION VALIDATION
Transition pairs: 115
Invalid or non-positive transition counts: 0
Negative average times: 0
Negative median times: 0
Rows with missing transition metrics: 0

CareFlow validation checks completed.


In [7]:
# CareFlow: Patient Pathway and Bottleneck Analysis

import pandas as pd

frequency = data["activity_frequency"].copy()
transitions = data["activity_times"].copy()
cases = data["patient_cases"].copy()

# Ensure numeric columns are numeric
frequency["total_occurrences"] = pd.to_numeric(
    frequency["total_occurrences"], errors="coerce"
)
frequency["unique_cases"] = pd.to_numeric(
    frequency["unique_cases"], errors="coerce"
)

for col in ["total_transitions", "average_minutes", "median_minutes"]:
    transitions[col] = pd.to_numeric(
        transitions[col], errors="coerce"
    )

# 1. Most frequent activities
print("=" * 65)
print("1. TOP 10 MOST FREQUENT ACTIVITIES")
print("=" * 65)

top_activities = frequency.sort_values(
    "total_occurrences", ascending=False
).head(10)

display(top_activities)

# 2. Most frequent activity-to-activity transitions
print("\n" + "=" * 65)
print("2. TOP 10 MOST FREQUENT TRANSITIONS")
print("=" * 65)

top_transitions = transitions.sort_values(
    "total_transitions", ascending=False
).head(10)

display(top_transitions[
    ["previous_activity", "concept:name",
     "total_transitions", "average_minutes", "median_minutes"]
])

# 3. Transitions with the longest average time gaps
print("\n" + "=" * 65)
print("3. TOP 10 LONGEST AVERAGE TRANSITION TIMES")
print("=" * 65)

longest_average = transitions.sort_values(
    "average_minutes", ascending=False
).head(10)

display(longest_average[
    ["previous_activity", "concept:name",
     "total_transitions", "average_minutes", "median_minutes"]
])

# 4. Transitions with the longest median time gaps
print("\n" + "=" * 65)
print("4. TOP 10 LONGEST MEDIAN TRANSITION TIMES")
print("=" * 65)

longest_median = transitions.sort_values(
    "median_minutes", ascending=False
).head(10)

display(longest_median[
    ["previous_activity", "concept:name",
     "total_transitions", "average_minutes", "median_minutes"]
])

# 5. Summary of visit durations
print("\n" + "=" * 65)
print("5. PATIENT VISIT DURATION SUMMARY")
print("=" * 65)

cases["treatment_duration_hours"] = pd.to_numeric(
    cases["treatment_duration_hours"], errors="coerce"
)

display(cases["treatment_duration_hours"].describe().to_frame(
    name="Duration (hours)"
))

print("\nPathway and bottleneck exploration completed.")

1. TOP 10 MOST FREQUENT ACTIVITIES


,concept:name,total_occurrences,unique_cases
0,Leucocytes,3373,1011
1,CRP,3257,1006
2,LacticAcid,1464,859
3,Admission NC,1180,799
4,ER Triage,1052,1049
5,ER Registration,1049,1049
6,ER Sepsis Triage,1048,1048
7,IV Antibiotics,822,822
8,IV Liquid,752,752
9,Release A,671,671



2. TOP 10 MOST FREQUENT TRANSITIONS


,previous_activity,concept:name,total_transitions,average_minutes,median_minutes
0,Leucocytes,CRP,1774,344.959977,0.000000
1,CRP,Leucocytes,1443,744.306306,0.000000
2,ER Registration,ER Triage,970,10.596942,7.908333
3,ER Triage,ER Sepsis Triage,904,2.903909,0.416667
4,CRP,LacticAcid,627,61.161085,0.000000
5,LacticAcid,Leucocytes,564,186.920213,0.000000
6,IV Liquid,IV Antibiotics,501,13.463074,0.100000
7,IV Antibiotics,Admission NC,489,64.881902,5.866667
8,Leucocytes,Leucocytes,453,2101.260486,1440.000000
9,Leucocytes,LacticAcid,435,97.174713,0.000000



3. TOP 10 LONGEST AVERAGE TRANSITION TIMES


,previous_activity,concept:name,total_transitions,average_minutes,median_minutes
114,Release E,Return ER,1,162308.333333,162308.333333
16,Release A,Return ER,276,118568.580254,68064.033333
81,Release C,Return ER,6,98143.616667,100381.183333
78,Release D,Return ER,10,80068.766667,30413.891667
110,Leucocytes,Return ER,1,16852.650000,16852.650000
87,Admission NC,Release C,4,12681.583333,13365.000000
102,Admission NC,Release D,1,11516.616667,11516.616667
80,Leucocytes,Release C,7,7772.142857,5940.000000
95,IV Antibiotics,Release A,2,6773.950000,6773.950000
70,CRP,Release C,13,6603.076923,5850.000000



4. TOP 10 LONGEST MEDIAN TRANSITION TIMES


,previous_activity,concept:name,total_transitions,average_minutes,median_minutes
114,Release E,Return ER,1,162308.333333,162308.333333
81,Release C,Return ER,6,98143.616667,100381.183333
16,Release A,Return ER,276,118568.580254,68064.033333
78,Release D,Return ER,10,80068.766667,30413.891667
110,Leucocytes,Return ER,1,16852.650000,16852.650000
87,Admission NC,Release C,4,12681.583333,13365.000000
102,Admission NC,Release D,1,11516.616667,11516.616667
95,IV Antibiotics,Release A,2,6773.950000,6773.950000
80,Leucocytes,Release C,7,7772.142857,5940.000000
96,Leucocytes,Release E,2,5925.000000,5925.000000



5. PATIENT VISIT DURATION SUMMARY


,Duration (hours)
count,1049.000000
mean,683.624075
std,1453.534689
min,0.033889
25%,15.296111
50%,128.144167
75%,432.311389
max,10135.774722



Pathway and bottleneck exploration completed.


In [8]:
# CareFlow: Investigate unusually long time gaps

# 1. All transitions leading to Return ER
print("1. TRANSITIONS LEADING TO RETURN ER")

return_er = transitions[
    transitions["concept:name"].astype(str).str.strip().eq("Return ER")
].copy()

return_er["average_days"] = return_er["average_minutes"] / 1440
return_er["median_days"] = return_er["median_minutes"] / 1440

display(
    return_er[
        ["previous_activity", "concept:name", "total_transitions",
         "average_days", "median_days"]
    ].sort_values("total_transitions", ascending=False)
)

# 2. Longest recorded cases
print("\n2. TEN LONGEST RECORDED CASE DURATIONS")

cases["treatment_duration_hours"] = pd.to_numeric(
    cases["treatment_duration_hours"], errors="coerce"
)

long_cases = cases.nlargest(10, "treatment_duration_hours").copy()
long_cases["duration_days"] = (
    long_cases["treatment_duration_hours"] / 24
)

display(
    long_cases[
        ["case:concept:name", "first_event_time", "last_event_time",
         "treatment_duration_hours", "duration_days"]
    ]
)

# 3. Duration percentiles for context
print("\n3. DURATION PERCENTILES (DAYS)")

print(
    (cases["treatment_duration_hours"] / 24)
    .quantile([0.50, 0.75, 0.90, 0.95, 0.99])
    .rename("duration_days")
)

1. TRANSITIONS LEADING TO RETURN ER


,previous_activity,concept:name,total_transitions,average_days,median_days
16,Release A,Return ER,276,82.339292,47.266690
78,Release D,Return ER,10,55.603310,21.120758
81,Release C,Return ER,6,68.155289,69.709155
110,Leucocytes,Return ER,1,11.703229,11.703229
114,Release E,Return ER,1,112.714120,112.714120



2. TEN LONGEST RECORDED CASE DURATIONS


,case:concept:name,first_event_time,last_event_time,treatment_duration_hours,duration_days
300,HG,2013-12-13 10:58:49+00:00,2015-02-08 18:45:18+00:00,10135.774722,422.323947
852,VAA,2014-02-05 13:09:05+00:00,2015-04-03 08:55:52+00:00,10123.779722,421.824155
1041,ZS,2013-11-14 10:58:51+00:00,2014-12-18 09:08:54+00:00,9574.167500,398.923646
484,LS,2013-11-18 16:24:56+00:00,2014-12-20 15:39:26+00:00,9527.241667,396.968403
814,UBA,2014-05-08 09:58:15+00:00,2015-05-30 09:14:55+00:00,9287.277778,386.969907
295,HDA,2014-01-04 19:49:48+00:00,2015-01-04 18:28:24+00:00,8758.643333,364.943472
17,AI,2013-12-19 18:47:57+00:00,2014-12-19 01:38:36+00:00,8742.844167,364.285174
478,LN,2014-02-18 21:28:35+00:00,2015-01-27 12:12:50+00:00,8222.737500,342.614062
419,KE,2013-12-16 14:30:19+00:00,2014-11-17 14:54:09+00:00,8064.397222,336.016551
23,AL,2014-04-07 16:13:06+00:00,2015-02-22 05:16:14+00:00,7693.052222,320.543843



3. DURATION PERCENTILES (DAYS)
0.50      5.339340
0.75     18.012975
0.90     92.747086
0.95    163.944319
0.99    298.809612
Name: duration_days, dtype: float64


Milestone 2 & 3 — Data Validation and Initial Analysis

Data quality validation

- Patient cases analyzed: 1,049
- Total events: 15,189
- Distinct activities: 16
- Activity-to-activity transition pairs: 115
- Duplicate case IDs: 0
- Event totals across summary files: matched
- Invalid timestamps and negative durations: 0
- Duration mismatches exceeding one minute: 0

Initial pathway findings

- Leucocytes and CRP were the most frequent activities.
- ER Registration → ER Triage occurred 970 times, with an average transition gap of approximately 10.6 minutes.
- ER Triage → ER Sepsis Triage occurred 904 times, with an average transition gap of approximately 2.9 minutes.
- Some transitions to Return ER showed long gaps and require interpretation as possible repeat visits rather than confirmed hospital bottlenecks.
- Recorded case durations were strongly right-skewed, so average duration should be interpreted alongside the median.

Next steps

- Validate the original XES event log against the summary CSV files.
- Generate a process map using PM4Py.
- Analyze common pathways, repeated activities, and potential bottlenecks.
- Prepare validated aggregate metrics for the Power BI dashboard.

Note: Findings are preliminary and based on the supplied CSV summaries. Raw event-log validation is pending.

In [9]:
from google.colab import files

uploaded_xes = files.upload()

Saving sepsis.xes.gz to sepsis.xes.gz


In [10]:
from pathlib import Path

for file in Path("/content").glob("sepsis*"):
    print("File found:", file.name)
    print("File size:", round(file.stat().st_size / 1024, 2), "KB")

File found: sepsis.xes.gz
File size: 197.76 KB


In [11]:
%pip install pm4py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 37.8 MB/s  0:00:00


In [12]:
import pm4py

print("PM4Py installed successfully!")
print("PM4Py version:", pm4py.__version__)



  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




PM4Py installed successfully!
PM4Py version: 2.7.23.12


In [13]:
import os

xes_path = "/content/sepsis.xes.gz"

if os.path.exists(xes_path):
    print("XES file found successfully!")
    print("File size (KB):", round(os.path.getsize(xes_path) / 1024, 2))
else:
    print("XES file not found. Please upload it again.")

XES file found successfully!
File size (KB): 197.76


In [14]:

import pm4py

xes_path = "/content/sepsis.xes.gz"

event_log = pm4py.read_xes(xes_path)

print("Event log loaded successfully!")
print("Object type:", type(event_log))
print("Total event records:", len(event_log))

/usr/local/lib/python3.13/dist-packages/pm4py/utils.py:1027: UserWarning: Install the optional requirement `r4pm` to import/export files faster. `rustxes` remains supported as a fallback.
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/1050 [00:00<?, ?it/s]

Event log loaded successfully!
Object type: <class 'pandas.core.frame.DataFrame'>
Total event records: 15214


In [15]:
print("Object type:", type(event_log))
print("Shape:", event_log.shape)

print("\nColumn names:")
print(event_log.columns.tolist())

print("\nFirst 5 rows:")
display(event_log.head())

Object type: <class 'pandas.core.frame.DataFrame'>
Shape: (15214, 32)

Column names:
['InfectionSuspected', 'org:group', 'DiagnosticBlood', 'DisfuncOrg', 'SIRSCritTachypnea', 'Hypotensie', 'SIRSCritHeartRate', 'Infusion', 'DiagnosticArtAstrup', 'concept:name', 'Age', 'DiagnosticIC', 'DiagnosticSputum', 'DiagnosticLiquor', 'DiagnosticOther', 'SIRSCriteria2OrMore', 'DiagnosticXthorax', 'SIRSCritTemperature', 'time:timestamp', 'DiagnosticUrinaryCulture', 'SIRSCritLeucos', 'Oligurie', 'DiagnosticLacticAcid', 'lifecycle:transition', 'Diagnose', 'Hypoxie', 'DiagnosticUrinarySediment', 'DiagnosticECG', 'case:concept:name', 'Leucocytes', 'CRP', 'LacticAcid']

First 5 rows:


,InfectionSuspected,org:group,DiagnosticBlood,DisfuncOrg,SIRSCritTachypnea,Hypotensie,SIRSCritHeartRate,Infusion,DiagnosticArtAstrup,concept:name,...,DiagnosticLacticAcid,lifecycle:transition,Diagnose,Hypoxie,DiagnosticUrinarySediment,DiagnosticECG,case:concept:name,Leucocytes,CRP,LacticAcid
0,True,A,True,True,True,True,True,True,True,ER Registration,...,True,complete,A,False,True,True,A,NaN,NaN,NaN
1,NaN,B,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Leucocytes,...,NaN,complete,NaN,NaN,NaN,NaN,A,9.6,NaN,NaN
2,NaN,B,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CRP,...,NaN,complete,NaN,NaN,NaN,NaN,A,NaN,21.0,NaN
3,NaN,B,NaN,NaN,NaN,NaN,NaN,NaN,NaN,LacticAcid,...,NaN,complete,NaN,NaN,NaN,NaN,A,NaN,NaN,2.2
4,NaN,C,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ER Triage,...,NaN,complete,NaN,NaN,NaN,NaN,A,NaN,NaN,NaN


In [16]:
# Validate the original CareFlow event log

print("Total event records:", len(event_log))

print(
    "Unique cases:",
    event_log["case:concept:name"].nunique()
)

print(
    "Distinct activities:",
    event_log["concept:name"].nunique()
)

print(
    "Missing case IDs:",
    event_log["case:concept:name"].isna().sum()
)

print(
    "Missing activity names:",
    event_log["concept:name"].isna().sum()
)

print(
    "Missing timestamps:",
    event_log["time:timestamp"].isna().sum()
)

print("\nTop 10 activities:")
display(
    event_log["concept:name"]
    .value_counts()
    .head(10)
    .rename_axis("Activity")
    .reset_index(name="Occurrences")
)

Total event records: 15214
Unique cases: 1050
Distinct activities: 16
Missing case IDs: 0
Missing activity names: 0
Missing timestamps: 0

Top 10 activities:


,Activity,Occurrences
0,Leucocytes,3383
1,CRP,3262
2,LacticAcid,1466
3,Admission NC,1182
4,ER Triage,1053
5,ER Registration,1050
6,ER Sepsis Triage,1049
7,IV Antibiotics,823
8,IV Liquid,753
9,Release A,671


In [17]:
import os
import pandas as pd
import zipfile
import glob

# Find CSV files available in Colab
csv_files = glob.glob("/content/**/*.csv", recursive=True)

print("CSV files found:")
for f in csv_files:
    print(f)

CSV files found:
/content/activity_frequency.csv
/content/activity_time_differences.csv
/content/patient_case_summary.csv
/content/sample_data/mnist_test.csv
/content/sample_data/mnist_train_small.csv
/content/sample_data/california_housing_test.csv
/content/sample_data/california_housing_train.csv


In [20]:

# Compare original XES data with the supplied CSV summaries
case_summary = pd.read_csv("/content/patient_case_summary.csv")
activity_summary = pd.read_csv("/content/activity_frequency.csv")

# Compare unique case IDs
xes_cases = set(event_log["case:concept:name"].unique())
csv_cases = set(case_summary["case:concept:name"].dropna().unique())

print("CASE COMPARISON")
print("Cases in XES:", len(xes_cases))
print("Cases in CSV:", len(csv_cases))
print("Cases only in XES:", len(xes_cases - csv_cases))
print("Cases only in CSV:", len(csv_cases - xes_cases))

# Compare total event records
print("\nEVENT COUNT COMPARISON")
print("XES event records:", len(event_log))
print("CSV summary event total:", case_summary["total_events"].sum())
print("Difference:", len(event_log) - case_summary["total_events"].sum())

# Compare activity counts
xes_activity_counts = event_log["concept:name"].value_counts()
csv_activity_counts = activity_summary.set_index("concept:name")["total_occurrences"]

activity_comparison = pd.concat(
    [
        xes_activity_counts.rename("XES_count"),
        csv_activity_counts.rename("CSV_count")
    ],
    axis=1
).fillna(0)

activity_comparison["Difference"] = (
    activity_comparison["XES_count"] - activity_comparison["CSV_count"]
)

print("\nACTIVITIES WITH DIFFERENT COUNTS")
display(activity_comparison[activity_comparison["Difference"] != 0])

CASE COMPARISON
Cases in XES: 1050
Cases in CSV: 1049
Cases only in XES: 1
Cases only in CSV: 0

EVENT COUNT COMPARISON
XES event records: 15214
CSV summary event total: 15189
Difference: 25

ACTIVITIES WITH DIFFERENT COUNTS


,XES_count,CSV_count,Difference
concept:name,,,
Leucocytes,3383,3373,10
CRP,3262,3257,5
LacticAcid,1466,1464,2
Admission NC,1182,1180,2
ER Triage,1053,1052,1
ER Registration,1050,1049,1
ER Sepsis Triage,1049,1048,1
IV Antibiotics,823,822,1
IV Liquid,753,752,1


In [21]:
import os
print(os.listdir("/content"))

['.config', 'activity_frequency.csv', 'sepsis.xes.gz', 'activity_time_differences.csv', 'patient_case_summary.csv', 'sample_data']


In [22]:
# Investigate the case missing from the CSV summary

missing_cases = xes_cases - csv_cases

missing_case_data = event_log[
    event_log["case:concept:name"].isin(missing_cases)
].copy()

print("MISSING CASE INVESTIGATION")
print("Cases missing from CSV:", len(missing_cases))
print("Events belonging to missing case(s):", len(missing_case_data))

print("\nActivity counts in missing case(s):")
display(
    missing_case_data["concept:name"]
    .value_counts()
    .rename("Event_Count")
    .to_frame()
)

# Compare event counts for matching cases

xes_case_counts = event_log.groupby("case:concept:name").size()

csv_case_counts = case_summary.set_index(
    "case:concept:name"
)["total_events"]

common_cases = xes_case_counts.index.intersection(
    csv_case_counts.index
)

case_count_comparison = pd.DataFrame({
    "XES_events": xes_case_counts.loc[common_cases],
    "CSV_events": csv_case_counts.loc[common_cases]
})

case_count_comparison["Difference"] = (
    case_count_comparison["XES_events"] -
    case_count_comparison["CSV_events"]
)

print("\nCOMPARISON OF MATCHING CASES")
print("Matching cases:", len(common_cases))
print(
    "Cases with different event counts:",
    (case_count_comparison["Difference"] != 0).sum()
)
print(
    "Total event-count difference in matching cases:",
    case_count_comparison["Difference"].sum()
)

MISSING CASE INVESTIGATION
Cases missing from CSV: 1
Events belonging to missing case(s): 24

Activity counts in missing case(s):


,Event_Count
concept:name,
Leucocytes,9
CRP,5
LacticAcid,2
Admission NC,2
ER Triage,1
ER Registration,1
IV Liquid,1
IV Antibiotics,1
ER Sepsis Triage,1



COMPARISON OF MATCHING CASES
Matching cases: 1049
Cases with different event counts: 1
Total event-count difference in matching cases: 1


In [23]:

# Investigate event-count differences among matching cases

mismatched_cases = case_count_comparison[
    case_count_comparison["Difference"] != 0
]

print("MATCHING CASES WITH DIFFERENT EVENT COUNTS")
print("Number of mismatched cases:", len(mismatched_cases))

print("\nEvent counts (case IDs intentionally not displayed):")
display(
    mismatched_cases[
        ["XES_events", "CSV_events", "Difference"]
    ].reset_index(drop=True)
)

print("\nFINAL EVENT COUNT CHECK")
print(
    "XES events across matching cases:",
    xes_case_counts.loc[common_cases].sum()
)
print(
    "CSV events across matching cases:",
    csv_case_counts.loc[common_cases].sum()
)
print(
    "Difference:",
    xes_case_counts.loc[common_cases].sum()
    - csv_case_counts.loc[common_cases].sum()
)

MATCHING CASES WITH DIFFERENT EVENT COUNTS
Number of mismatched cases: 1

Event counts (case IDs intentionally not displayed):


,XES_events,CSV_events,Difference
0,48,47,1



FINAL EVENT COUNT CHECK
XES events across matching cases: 15190
CSV events across matching cases: 15189
Difference: 1


## XES and CSV Data Reconciliation

The original XES event log was compared with the three CSV summary files provided for the CareFlow project.

### Original XES Dataset
- Total event records: 15,214
- Unique cases: 1,050
- Distinct activities: 16

### Supplied CSV Summaries
- Total event records: 15,189
- Unique cases: 1,049
- Distinct activities: 16

### Identified Discrepancies
1. One case containing 24 events is absent from the CSV case summary.
2. Among the 1,049 matching cases, one case has 48 events in XES and 47 events in the CSV summary.
3. The remaining 1,048 matching cases have identical event counts.
4. The activity-level comparison shows a total difference of 25 events. The missing case accounts for 24 events, and the remaining difference is one Leucocytes event.

### Conclusion
The numerical difference of 25 events between the original XES dataset and the supplied CSV summaries has been accounted for.

The reason for the missing case and the additional missing event has not yet been established. The original XES dataset and supplied CSV summaries are therefore retained separately, and further analysis must clearly identify which data source is being used.